# Chat Format and SFT


In [1]:
import sys
from pathlib import Path

ROOT = Path('/Users/montekkundan/Developer/ML/llm')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pathlib import Path
import copy
from course_tools import DEFAULT_CHAT_MESSAGES, build_demo_bundle, format_messages, generate_text, train_model

LECTURE_NOTE_PATH = Path('/Users/montekkundan/Library/Mobile Documents/iCloud~md~obsidian/Documents/lectures/LLM/concepts/Chat Format and SFT.md')
print(LECTURE_NOTE_PATH)


/Users/montekkundan/Library/Mobile Documents/iCloud~md~obsidian/Documents/lectures/LLM/concepts/Chat Format and SFT.md


## Demo A: a conversation is flattened into one causal token stream


In [2]:
formatted = format_messages(DEFAULT_CHAT_MESSAGES + [{'role': 'assistant', 'content': 'Self-attention lets tokens mix information from relevant context.'}])
print(formatted)


<|system|>
You are a compact teaching assistant for an LLM course.
<|user|>
Explain self-attention in three short sentences.
<|assistant|>
Self-attention lets tokens mix information from relevant context.



## Demo B: base model vs SFT model


In [3]:
bundle = build_demo_bundle(steps=20)
tokenizer = bundle['tokenizer']
prompt = format_messages(DEFAULT_CHAT_MESSAGES, add_assistant_prompt=True)
base_reply = generate_text(bundle['model'], tokenizer, prompt, max_new_tokens=40, temperature=0.0)
print('base reply:', repr(base_reply))

sft_model = copy.deepcopy(bundle['model'])
sft_text = '\n'.join([
    format_messages(DEFAULT_CHAT_MESSAGES + [{'role': 'assistant', 'content': 'Self-attention creates a weighted summary of earlier tokens.'}]),
] * 8)
train_model(sft_model, tokenizer, train_text=sft_text, eval_text=sft_text, steps=10, learning_rate=2e-3, batch_size=4)
sft_reply = generate_text(sft_model, tokenizer, prompt, max_new_tokens=40, temperature=0.0)
print('sft reply :', repr(sft_reply))


base reply: '\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n'
sft reply : '\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n'


## Demo C: the architecture stayed the same, the training distribution changed


In [4]:
print(type(bundle['model']).__name__)
print(type(sft_model).__name__)
print('same architecture, different parameters after extra chat-style training')


TinyTransformerLM
TinyTransformerLM
same architecture, different parameters after extra chat-style training


## Rasbt and nanochat


**RASBT**
- https://github.com/rasbt/LLMs-from-scratch/blob/main/ch07/01_main-chapter-code/gpt_instruction_finetuning.py
**NANOCHAT**
- https://github.com/karpathy/nanochat/blob/master/scripts/chat_sft.py
- https://github.com/karpathy/nanochat/blob/master/nanochat/tokenizer.py
